In [1]:
%pylab inline

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [ ]:
# import dataiku
# from dataiku import pandasutils as pdu
import pandas as pd
import os

In [ ]:
# Read the dataset as a Pandas dataframe in memory
# Ruta relativa al archivo de datos
DATA_PATH = "C:\Users\Javi\Desktop\A 5º Mat-Info\TFG Mates\Mathematics-Dissertation-Survival-Modeling-Javier\data\processed\icu_master.csv"

def get_data(filepath=DATA_PATH):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"No se encuentra el archivo en {filepath}. ¿Has descargado los datos?")
    
    # Cargar datos
    df = pd.read_csv(filepath)
    return df

# Uso
df = get_data()

#dataset_icu_master = dataiku.Dataset("icu_master")
#df = dataset_icu_master.get_dataframe(limit=100000)

In [4]:
# Get some simple descriptive statistics
pdu.audit(df)

,_a_variable,_b_data_type,_c_cardinality,_d_missings,_e_sample_values
0,subject_id,int64,36931,0,"[10584718, 14671248]"
1,hadm_id,int64,42710,0,"[23485217, 21908050]"
2,stay_id,int64,43386,0,"[36809832, 34687545]"
3,intime,object,43383,0,"[2165-02-27 21:41:10, 2125-11-09 22:48:10]"
4,outtime,object,43385,0,"[2165-03-06 11:09:58, 2125-11-11 15:21:27]"
5,admittime,object,42557,0,"[2165-02-12 15:41:00, 2125-11-05 18:28:00]"
6,dischtime,object,42578,0,"[2165-03-06 08:20:00, 2125-11-12 16:30:00]"
7,deathtime,object,5737,37478,"[2165-03-06 08:20:00, nan]"
8,hospital_expire_flag,int64,2,0,"[1, 0]"
9,sex,object,2,0,"[M, F]"


# Procesamiento de Datos

## Variables and Initial Dataframe

In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# -----------------------------
# Definimos columnas a usar
# -----------------------------
id_features = ["subject_id", "hadm_id", "stay_id"]

categorical_features = [
    "sex",
    "unit_type",
    "admission_location",
    "race",
]
# cuidado porque hemos tomado variables gcs como númericas, discutirlo
numerical_features = [
    "age",
    "height",
    "weight",
    "gcs_motor",
    "gcs_verbal",
    "hour_of_admission",
    # puedes añadir "time_since_admission" si quieres, pero cuidado con interpretar
]

# Outcome de supervivencia
time_col = "time"
event_col = "event"

# Dataset de trabajo (sin IDs)
df_model = df[categorical_features + numerical_features + [time_col, event_col]].copy()

# considerar añadir mascara ya que me doy cuenta de que hay 35 personas con time negativo
mask_valid = df_model["time"] > 0
df_model = df_model[mask_valid].copy()


## Splits and Preprocessing

In [29]:
from sklearn.model_selection import train_test_split

def make_splits(df_model, event_col, seed):
    """Devuelve train_df, val_df, test_df con 60/20/20, estratificados por event."""
    train_df, temp_df = train_test_split(
        df_model,
        test_size=0.4,           # 60% train, 40% resto
        stratify=df_model[event_col],
        random_state=seed,
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,           # 20% val, 20% test
        stratify=temp_df[event_col],
        random_state=seed,
    )
    return train_df, val_df, test_df



In [30]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from scipy import sparse
import numpy as np

def build_preprocessor(categorical_features, numerical_features):
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), numerical_features),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), categorical_features),
        ]
    )
    return preprocessor

def to_dense(X):
    return X.toarray() if sparse.issparse(X) else X

def preprocess_split(train_df, val_df, test_df,
                     categorical_features, numerical_features,
                     time_col="time", event_col="event"):

    preprocessor = build_preprocessor(categorical_features, numerical_features)
    
    
    # Solo se transforma (transformación normal) el dataset de train, favorece entrenamiento de la red neuronal
    X_train = preprocessor.fit_transform(train_df)
    
    X_val   = preprocessor.transform(val_df)
    X_test  = preprocessor.transform(test_df)

    X_train_dense = to_dense(X_train).astype("float32")
    X_val_dense   = to_dense(X_val).astype("float32")
    X_test_dense  = to_dense(X_test).astype("float32")
    
    # Guardamos datos de target de val y test

    y_train_time  = train_df[time_col].values
    y_train_event = train_df[event_col].values
    y_val_time    = val_df[time_col].values
    y_val_event   = val_df[event_col].values
    y_test_time   = test_df[time_col].values
    y_test_event  = test_df[event_col].values

    return (X_train_dense, X_val_dense, X_test_dense,
            y_train_time, y_train_event,
            y_val_time, y_val_event,
            y_test_time, y_test_event)

In [31]:
train_df, val_df, test_df = make_splits(df_model, event_col="event", seed=42)


(X_train_d, X_val_d, X_test_d,
 y_train_time, y_train_event,
 y_val_time,   y_val_event,
 y_test_time,  y_test_event) = preprocess_split(
                                train_df, val_df, test_df,
                                categorical_features, numerical_features,
                                time_col="time", event_col="event"
                                )

In [32]:
import numpy as np

# Cortes para PWE en días: 0-1, 1-2, ..., 9-10
breaks = np.linspace(0.0, 10.0, 11)  # 11 puntos → 10 intervalos
print(breaks)


[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10.]


In [33]:

def expand_to_pwe(X, time, event, breaks):
    """
    X: np.array (n, p) matriz de covariables ya preprocesadas
    time: np.array (n,) tiempo en días (0-10)
    event: np.array (n,) 1=muere antes o en 10d, 0=censura
    breaks: np.array de puntos de corte (ej: [0,1,...,10])

    Devuelve:
      df_pwe: DataFrame con columnas:
        - 'y'   exposición en el intervalo
        - 'd'   indicador de evento en el intervalo
        - 'k'   índice de intervalo (0,...,K-1)
        - columnas x_0...x_{p-1} covariables
    """
    n, p = X.shape
    K = len(breaks) - 1
    rows = []

    for i in range(n): ## rango de todos los pacientes como 0,..., 43385
        t_i = time[i]
        e_i = event[i]
        x_i = X[i, :]

        for k in range(K):
            start = breaks[k]
            end = breaks[k+1]

            # Si el paciente ya "sale" antes de este intervalo, no aportamos nada
            if t_i <= start:
                break  # ya no está en riesgo en intervalos posteriores

            # tiempo en riesgo en este intervalo
            y_ik = min(t_i, end) - start
            if y_ik <= 0:
                continue  # por seguridad

            # evento en este intervalo: sólo si muere aquí
            d_ik = 1 if (e_i == 1 and t_i <= end) and (t_i > start) else 0

            row = {
                "id": i, ## guardar id del paciente para pruebas
                "y": y_ik,
                "d": d_ik,
                "k": k,
            }
            # añadir covariables numéricas como x_0, x_1, ...
            for j in range(p):
                row[f"x{j}"] = x_i[j]

            rows.append(row)

    df_pwe = pd.DataFrame(rows)
    return df_pwe


In [34]:
# p = nº de columnas de X después de preprocesar
p = X_train_d.shape[1]
print("Nº de features (p):", p)

train_pwe = expand_to_pwe(X_train_d, y_train_time, y_train_event, breaks)
val_pwe   = expand_to_pwe(X_val_d,   y_val_time,   y_val_event,   breaks)
test_pwe  = expand_to_pwe(X_test_d,  y_test_time,  y_test_event,  breaks)

print("train_pwe shape:", train_pwe.shape)
print("val_pwe shape:  ", val_pwe.shape)
print("test_pwe shape: ", test_pwe.shape)

display(train_pwe.head(10000))


Nº de features (p): 68
train_pwe shape: (185416, 72)
val_pwe shape:   (61434, 72)
test_pwe shape:  (61501, 72)


,id,y,d,k,x0,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,x11,x12,x13,x14,x15,x16,x17,x18,x19,x20,x21,x22,x23,x24,x25,x26,x27,x28,x29,x30,x31,x32,x33,x34,x35,x36,x37,x38,x39,x40,x41,x42,x43,x44,x45,x46,x47,x48,x49,x50,x51,x52,x53,x54,x55,x56,x57,x58,x59,x60,x61,x62,x63,x64,x65,x66,x67
0,0,1.0,0,0,-0.516641,-0.932787,1.231115,0.563568,0.954128,0.075391,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,0,1.0,0,1,-0.516641,-0.932787,1.231115,0.563568,0.954128,0.075391,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,0,1.0,0,2,-0.516641,-0.932787,1.231115,0.563568,0.954128,0.075391,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,0,1.0,0,3,-0.516641,-0.932787,1.231115,0.563568,0.954128,0.075391,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,0,1.0,0,4,-0.516641,-0.932787,1.231115,0.563568,0.954128,0.075391,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,1398,1.0,0,0,-0.012183,-1.104613,0.335295,0.563568,0.954128,-0.692668,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
9996,1398,1.0,0,1,-0.012183,-1.104613,0.335295,0.563568,0.954128,-0.692668,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
9997,1398,1.0,0,2,-0.012183,-1.104613,0.335295,0.563568,0.954128,-0.692668,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
9998,1398,1.0,0,3,-0.012183,-1.104613,0.335295,0.563568,0.954128,-0.692668,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


## Checks del nuevo dataframe con intervalos

In [35]:
import numpy as np

def quick_check_pwe(df_pwe, name, breaks):
    print(f"\n=== {name} PWE ===")
    print("Shape:", df_pwe.shape)
    print("Columnas:", df_pwe.columns.tolist()[:10], "...")

    # 1) d debe ser 0 ó 1
    print("\nConteos de d:")
    print(df_pwe["d"].value_counts(dropna=False))

    # 2) y siempre > 0
    n_y_leq0 = (df_pwe["y"] <= 0).sum()
    print("\nFilas con y <= 0:", n_y_leq0)

    # 3) y no puede ser mayor que la longitud máxima de un intervalo
    interval_lengths = np.diff(breaks)
    max_len = interval_lengths.max()
    n_y_too_big = (df_pwe["y"] > max_len + 1e-8).sum()
    print("Filas con y > longitud de intervalo:", n_y_too_big)

    print("\nDescriptivos de y:")
    print(df_pwe["y"].describe())
    

quick_check_pwe(train_pwe, "TRAIN", breaks)
quick_check_pwe(val_pwe,   "VAL",   breaks)
quick_check_pwe(test_pwe,  "TEST",  breaks)



=== TRAIN PWE ===
Shape: (185416, 72)
Columnas: ['id', 'y', 'd', 'k', 'x0', 'x1', 'x2', 'x3', 'x4', 'x5'] ...

Conteos de d:
0    183089
1      2327
Name: d, dtype: int64

Filas con y <= 0: 0
Filas con y > longitud de intervalo: 0

Descriptivos de y:
count    185416.000000
mean          0.952448
std           0.175387
min           0.041667
25%           1.000000
50%           1.000000
75%           1.000000
max           1.000000
Name: y, dtype: float64

=== VAL PWE ===
Shape: (61434, 72)
Columnas: ['id', 'y', 'd', 'k', 'x0', 'x1', 'x2', 'x3', 'x4', 'x5'] ...

Conteos de d:
0    60658
1      776
Name: d, dtype: int64

Filas con y <= 0: 0
Filas con y > longitud de intervalo: 0

Descriptivos de y:
count    61434.000000
mean         0.952614
std          0.175033
min          0.041667
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000
Name: y, dtype: float64

=== TEST PWE ===
Shape: (61501, 72)
Columnas: ['id', 'y', 'd', 'k', 'x0', 'x1', 'x2', 'x3', 

In [39]:
def check_by_patient(df_pwe, time, event, name):
    print(f"\n=== Check por paciente: {name} ===")

    # suma de tiempos y eventos por id
    time_from_pwe = df_pwe.groupby("id")["y"].sum().sort_index().values
    event_from_pwe = df_pwe.groupby("id")["d"].sum().sort_index().values
    
    # vemos que no coinciden con los datasets originales, esto pasa porque filtramos los que tienen tim enegativo al expandir
    # en la funcion expand_to_pwe, no se pueden hacer comparaciones con estos dataset o
    print("  Coinciden nº de pacientes?",
          len(time_from_pwe) == len(time) == len(event))
    
     # diferencias
    diff_time = np.abs(time_from_pwe - time)
    diff_event = np.abs(event_from_pwe - event)

    print("  Max |diff time|:", diff_time.max())
    print("  Valores únicos de suma(d) por paciente:", np.unique(event_from_pwe))
    
    ## hacer pruebas para ver si cuadra
    print("  Valores censurados en 10 días: ", sum((time_from_pwe >= 10.).astype(int)))
    print("  Número de pacientes que sufren evento: ", sum((event_from_pwe > 0.).astype(int)))

    # cuántos pacientes tienen suma(d) > 1 (no debería pasar)
    print("  Pacientes con más de un evento:",
          (event_from_pwe > 1).sum())

check_by_patient(train_pwe, y_train_time, y_train_event, "TRAIN")
check_by_patient(val_pwe,   y_val_time,   y_val_event,   "VAL")
check_by_patient(test_pwe,  y_test_time,  y_test_event,  "TEST")



=== Check por paciente: TRAIN ===
  Coinciden nº de pacientes? True
  Max |diff time|: 0.0
  Valores únicos de suma(d) por paciente: [0 1]
  Valores censurados en 10 días:  8761
  Número de pacientes que sufren evento:  2327
  Pacientes con más de un evento: 0

=== Check por paciente: VAL ===
  Coinciden nº de pacientes? True
  Max |diff time|: 0.0
  Valores únicos de suma(d) por paciente: [0 1]
  Valores censurados en 10 días:  2938
  Número de pacientes que sufren evento:  776
  Pacientes con más de un evento: 0

=== Check por paciente: TEST ===
  Coinciden nº de pacientes? True
  Max |diff time|: 0.0
  Valores únicos de suma(d) por paciente: [0 1]
  Valores censurados en 10 días:  2862
  Número de pacientes que sufren evento:  776
  Pacientes con más de un evento: 0


# Modelizar

## 0. Idea del Modelo

Para cada paciente-intervalo (i,k):

* y_ik = tiempo en riesgo en ese intervalo (en días).

* d_ik = nº de eventos en ese intervalo (0 o 1).

* x_i = covariables (las mismas en todos sus intervalos).

Modelo:

* dik​∼Poisson(μik​), 
* μik​=λk​(xi​)yik​, 
* logλk​(xi​)=αk​+β⊤xi​. 

En código:

* log_mu = log(y_ik) + alpha_k + beta^T x_i.


## 1. Creamos Datasets y Loaders

In [41]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class PWEDataset(Dataset):
    def __init__(self, df_pwe, p):
        # guardamos tensores
        self.y  = torch.tensor(df_pwe["y"].values, dtype=torch.float32)
        self.d  = torch.tensor(df_pwe["d"].values, dtype=torch.float32)
        self.k  = torch.tensor(df_pwe["k"].values, dtype=torch.long)  # índice de intervalo
        self.id = torch.tensor(df_pwe["id"].values, dtype=torch.long)

        X_cols = [f"x{j}" for j in range(p)]
        self.X = torch.tensor(df_pwe[X_cols].values, dtype=torch.float32)

        # precomputamos log(y) para usarlo como offset
        eps = 1e-8
        self.log_y = torch.log(self.y + eps)

    def __len__(self):
        return len(self.d)

    def __getitem__(self, idx):
        return (
            self.X[idx],
            self.k[idx],
            self.log_y[idx],
            self.d[idx],
            self.id[idx],
        )


    
p = X_train_d.shape[1]
K = len(breaks) - 1

train_dataset = PWEDataset(train_pwe, p)
val_dataset   = PWEDataset(val_pwe,   p)
test_dataset  = PWEDataset(test_pwe,  p)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=4096, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=4096, shuffle=False)


## 2. Definimos el Modelo PWE en Pytorch

In [42]:
import torch.nn as nn

class PWEPoisson(nn.Module):
    def __init__(self, p, K):
        super().__init__()
        # β: vector de coeficientes (p x 1)
        self.beta_layer = nn.Linear(p, 1, bias=False)
        # α_k: efecto específico de intervalo (K parámetros)
        self.alpha = nn.Parameter(torch.zeros(K))

    def forward(self, X, k, log_y):
        """
        X:  (N, p)
        k:  (N,) índices de intervalo [0,...,K-1]
        log_y: (N,) log del tiempo en riesgo
        """
        lin = self.beta_layer(X).squeeze(-1)   # β^T x
        alpha_k = self.alpha[k]               # selecciona α_k para cada fila
        log_mu = log_y + alpha_k + lin        # log μ_ik
        return log_mu

    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_pwe = PWEPoisson(p=p, K=K).to(device)

# función de perdida de Poisson ya la tenemos en Pytorch
criterion = nn.PoissonNLLLoss(log_input=True, full=False, reduction="mean")
optimizer = torch.optim.Adam(model_pwe.parameters(), lr=1e-3)


## 3. Bucle de entrenamiento

In [43]:
def train_pwe_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    n_batches = 0

    for X, k, log_y, d, _ in loader:
        X     = X.to(device)
        k     = k.to(device)
        log_y = log_y.to(device)
        d     = d.to(device)

        optimizer.zero_grad()
        log_mu = model(X, k, log_y)
        loss = criterion(log_mu, d)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1

    return total_loss / n_batches


def eval_pwe_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    n_batches = 0

    with torch.no_grad():
        for X, k, log_y, d, _ in loader:
            X     = X.to(device)
            k     = k.to(device)
            log_y = log_y.to(device)
            d     = d.to(device)

            log_mu = model(X, k, log_y)
            loss = criterion(log_mu, d)

            total_loss += loss.item()
            n_batches  += 1

    return total_loss / n_batches


In [44]:
n_epochs = 20

for epoch in range(1, n_epochs+1):
    train_loss = train_pwe_epoch(model_pwe, train_loader, optimizer, criterion)
    val_loss   = eval_pwe_epoch(model_pwe,   val_loader,   criterion)

    print(f"Epoch {epoch:02d} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")


Epoch 01 | Train loss: 0.6614 | Val loss: 0.4598
Epoch 02 | Train loss: 0.3601 | Val loss: 0.2856
Epoch 03 | Train loss: 0.2402 | Val loss: 0.2049
Epoch 04 | Train loss: 0.1799 | Val loss: 0.1610
Epoch 05 | Train loss: 0.1457 | Val loss: 0.1348
Epoch 06 | Train loss: 0.1251 | Val loss: 0.1180
Epoch 07 | Train loss: 0.1116 | Val loss: 0.1067
Epoch 08 | Train loss: 0.1018 | Val loss: 0.0988
Epoch 09 | Train loss: 0.0945 | Val loss: 0.0931
Epoch 10 | Train loss: 0.0895 | Val loss: 0.0889
Epoch 11 | Train loss: 0.0861 | Val loss: 0.0858
Epoch 12 | Train loss: 0.0833 | Val loss: 0.0834
Epoch 13 | Train loss: 0.0810 | Val loss: 0.0815
Epoch 14 | Train loss: 0.0793 | Val loss: 0.0801
Epoch 15 | Train loss: 0.0780 | Val loss: 0.0790
Epoch 16 | Train loss: 0.0776 | Val loss: 0.0781
Epoch 17 | Train loss: 0.0762 | Val loss: 0.0773
Epoch 18 | Train loss: 0.0755 | Val loss: 0.0768
Epoch 19 | Train loss: 0.0752 | Val loss: 0.0763
Epoch 20 | Train loss: 0.0748 | Val loss: 0.0759


# Métricas

## C-Index

In [ ]:
import torch
import numpy as np

# primero tenemos que dar un risk score a cada paciente

def get_pwe_risk_scores(model_pwe, X_np, device):
    """
    Devuelve β^T x para cada paciente (vector de riesgo).
    """
    model_pwe.eval()
    X_t = torch.tensor(X_np, dtype=torch.float32, device=device)
    with torch.no_grad():
        risk = model_pwe.beta_layer(X_t).squeeze(-1)  # (n_test,)
    return risk.cpu().numpy()


# función harrel c-index a mano para nuestro caso
def c_index_harrell(time, event, risk):
    """
    time:  array (n,)
    event: array (n,) 1=evento, 0=censura
    risk:  array (n,) score (mayor = más riesgo)

    Devuelve: C-index
    """
    n = len(time)
    concordant = 0.0
    permissible = 0.0

    for i in range(n):
        for j in range(n):
            # par comparable: i tiene evento antes que j
            if time[i] < time[j] and event[i] == 1:
                permissible += 1
                if risk[i] > risk[j]:
                    concordant += 1
                elif risk[i] == risk[j]:
                    concordant += 0.5

    if permissible == 0:
        return np.nan
    return concordant / permissible

# Aplicar la métrica en test

# risk scores de test
risk_test = get_pwe_risk_scores(model_pwe, X_test_d, device)

# C-index en test
cindex_test = c_index_harrell(y_test_time, y_test_event, risk_test)
print("PWE - Test C-index:", cindex_test)


PWE - Test C-index: 0.7543438512744154


## Curvas de Supervivencia

In [46]:
# Baseline acumulada en un grid
def baseline_B(model_pwe, breaks, time_grid, device):
    """
    Calcula B(t) = sum_k exp(alpha_k) * Delta_k(t) para cada t del grid.
    """
    alpha = model_pwe.alpha.detach().cpu().numpy()  # (K,)
    K = len(alpha)
    B = []

    for t in time_grid:
        Bt = 0.0
        for k in range(K):
            start = breaks[k]
            end   = breaks[k+1]
            if t > start:
                dt = min(t, end) - start
                if dt > 0:
                    Bt += np.exp(alpha[k]) * dt
        B.append(Bt)
    return np.array(B)  # (len(time_grid),)

# Curvas de supervivencia usando la B(t) generada arriba
def pwe_survival_curves(model_pwe, X_np, breaks, t_max=10.0, n_times=100, device=None):
    """
    Devuelve:
      time_grid: (T,)
      S_pred:    (n_pacientes, T) con S(t|x_i)
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # grid de tiempos
    time_grid = np.linspace(0.0, t_max, n_times)

    # baseline acumulada B(t)
    B_t = baseline_B(model_pwe, breaks, time_grid, device)  # (T,)

    # riesgo por paciente r_i = exp(β^T x_i)
    model_pwe.eval()
    X_t = torch.tensor(X_np, dtype=torch.float32, device=device)
    with torch.no_grad():
        lin = model_pwe.beta_layer(X_t).squeeze(-1)  # β^T x
    r = torch.exp(lin).cpu().numpy()  # (n,)

    # S_i(t) = exp( - r_i * B(t) )
    # vectorizado: B_t[None, :] * r[:, None]
    Lambda = r[:, None] * B_t[None, :]  # (n, T)
    S_pred = np.exp(-Lambda)

    return time_grid, S_pred

# Uso para test
time_grid, S_pwe_test = pwe_survival_curves(
    model_pwe,
    X_test_d,
    breaks,
    t_max=10.0,
    n_times=100,
    device=device
)
print("S_pwe_test shape:", S_pwe_test.shape)  # (n_test, 100)


S_pwe_test shape: (8670, 100)


## IBS e IBLL

In [48]:
# 1) Kaplan-meier de censura como en DeepSurv
def km_censoring_survival(time, event):
    """
    time: tiempos de seguimiento
    event: 1=evento muerte, 0=censura
    Devuelve:
      unique_times: tiempos únicos ordenados
      G:            G(t) = P(C > t)
    """
    event_cens = 1 - event  # 1 si censura
    order = np.argsort(time)
    t_ord = time[order]
    e_ord = event_cens[order]

    unique_times = np.unique(t_ord)
    n = len(t_ord)
    G = []
    surv = 1.0
    at_risk_idx = 0

    for t_k in unique_times:
        at_risk = n - at_risk_idx
        d_k = np.sum((t_ord == t_k) & (e_ord == 1))
        if at_risk > 0:
            surv *= (1 - d_k / at_risk)
        G.append(surv)
        at_risk_idx = np.searchsorted(t_ord, t_k, side="right")

    return unique_times, np.array(G)

def step_func(times, values, t_eval):
    """
    times, values: definición de la función escalonada (KM).
    t_eval: array de tiempos donde evaluar (right-continuous).
    """
    idx = np.searchsorted(times, t_eval, side="right") - 1
    idx = np.clip(idx, 0, len(values)-1)
    return values[idx]


# 2) Calcular IBS e IBLL 

def compute_ibs_ibll(time, event, time_grid, S_pred):
    """
    time:      (n,)
    event:     (n,)
    time_grid: (T,)
    S_pred:    (n, T) supervivencias S_hat(t|x_i)

    Devuelve:
      IBS, IBLL  (IBLL será negativa; -IBLL positivo)
    """
    n, T = S_pred.shape

    # KM de censura
    tG, G = km_censoring_survival(time, event)

    BS_t = []
    BLL_t = []

    for j, t in enumerate(time_grid):
        S_t = S_pred[:, j]

        # indicadores
        I_event = (time <= t) & (event == 1)
        I_surv  = (time > t)

        # G en tiempos
        G_Ti = step_func(tG, G, time)      # G(T_i- aprox G(T_i))
        G_t  = step_func(tG, G, np.array([t]))[0]

        # Brier IPCW
        term1 = ((1 - S_t)**2) * I_event / np.maximum(G_Ti, 1e-6)
        term2 = (S_t**2)       * I_surv  / np.maximum(G_t,  1e-6)
        BS = (np.sum(term1) + np.sum(term2)) / n
        BS_t.append(BS)

        # Binomial log-likelihood IPCW
        # (ojo: logs → IBLL será negativa)
        term1_ll = np.log(1 - S_t + 1e-8) * I_event / np.maximum(G_Ti, 1e-6)
        term2_ll = np.log(S_t + 1e-8)     * I_surv  / np.maximum(G_t,  1e-6)
        BLL = (np.sum(term1_ll) + np.sum(term2_ll)) / n
        BLL_t.append(BLL)

    BS_t  = np.array(BS_t)
    BLL_t = np.array(BLL_t)

    t_max = time_grid[-1]
    IBS  = np.trapz(BS_t,  time_grid) / t_max
    IBLL = np.trapz(BLL_t, time_grid) / t_max

    return IBS, IBLL

# 3) Mostrar para test

IBS_pwe_test, IBLL_pwe_test = compute_ibs_ibll(
    y_test_time,
    y_test_event,
    time_grid,
    S_pwe_test
)

print(f"PWE - Test IBS : {IBS_pwe_test:.4f}")
print(f"PWE - Test IBLL: {IBLL_pwe_test:.4f}")



PWE - Test IBS : 0.7929
PWE - Test IBLL: -0.2196
